# MNIST ZyNet Model Creation

This notebook documents the **model-development workflow** used to create the MNIST model later deployed on the FPGA.

It intentionally stops **before Vivado project construction and hardware integration**. The FPGA implementation eventually diverged from the original ZyNet tutorial because the research design incorporated the OV7670 camera and custom image-processing/storage logic. Those implementation details belong with the FPGA hardware branch rather than with the model-development materials.

## Scope

This notebook covers:

1. MNIST dataset loading
2. activation-function choice
3. final network architecture
4. model training
5. saving and reloading the trained model
6. software validation of the trained model
7. threshold/binary-input validation used to inform the camera implementation
8. preparation of trained parameters for ZyNet
9. the pretrained ZyNet model definition
10. verified changes from the original ZyNet sample workflow
11. final model artifacts and reproducibility notes

### Reference resources

- ZyNet repository: <https://github.com/dsdnu/zynet>
- Original ZyNet MNIST sample: <https://github.com/dsdnu/zynet/blob/master/sample/mnistZyNet.py>
- MNIST dataset site: <https://yann.lecun.com/exdb/mnist/>

The training code also uses the `network2.py`/`mnist_loader.py` style associated with Michael Nielsen's *Neural Networks and Deep Learning* examples.


## Files expected in the same directory

For the simplest GitHub layout, place this notebook and its supporting files in the same directory:

```text
MNIST_ZyNet_Model_Creation.ipynb
mnist_loader.py
network2.py
trainNN.py
genWegitsAndBias.py
mnistZyNet.py
mnistZyNet2.py
test_binary_mnist.py
convert.py
mnist.pkl.gz
WeigntsAndBiases.txt
weights.mem
biases.mem

mnist_sigmoid_activation.png
mnist_historical_training_script.png
mnist_training_results.png
mnist_saved_model_file.png
```

The four PNG filenames above are the names used by the Markdown cells in this notebook.

> **Important:** `weights.mem` and `biases.mem` are final FPGA parameter artifacts. The uploaded `genWegitsAndBias.py` does **not** directly create those two aggregate `.mem` files; it creates per-neuron `.mif` files and C header arrays. Their later packaging for the FPGA parameter loader should therefore be documented with the FPGA implementation unless the exact generating script is also retained.


## 1. Historical training notes

The original presentation retained from the early FPGA work documents the training sequence:

- edit `network2.py` if changing between sigmoid and ReLU;
- keep sigmoid for this model;
- edit `trainNN.py` to select the network size and training settings;
- run `trainNN.py`;
- use the generated weights-and-biases file as the pretrained model input for ZyNet.

The screenshots below are retained as **historical evidence**. Some show an earlier model configuration and should not be treated as the authoritative final architecture. The final scripts and saved model later in this notebook are the authoritative configuration.


### Sigmoid activation retained

![Sigmoid activation used in network2.py](mnist_sigmoid_activation.png)

The `network2.py` uses the sigmoid activation:

```python
def sigmoid(z):
    return 1.0/(1.0+np.exp(-z))

def sigmoid_prime(z):
    return sigmoid(z)*(1-sigmoid(z))
```

Commented ReLU alternatives remain in the file, showing that activation choice was considered but sigmoid was used for the working model.


### Historical training-script screenshot

![Historical training script](mnist_historical_training_script.png)

This screenshot records an earlier training configuration. It is useful as provenance, but the final `trainNN.py` below supersedes the layer sizes visible in this screenshot.


### Historical training output

![Historical training output](mnist_training_results.png)

The screenshot shows the end of a 100-epoch training run, including validation results for epochs 94--99. The final visible epoch reports **9602 / 10000** on the evaluation data.

This screenshot demonstrates the original training process, but it is not used as the final accuracy value for the final model. The saved model is evaluated independently later in this notebook.


## 2. Load the MNIST dataset

The `mnist_loader.py` reads `mnist.pkl.gz`, reshapes each image to a 784-element column vector, one-hot encodes training labels, and returns training, validation, and test iterables.

A compatibility change in the loader is the explicit encoding argument used when unpickling the older MNIST archive:

```python
training_data, validation_data, test_data = cPickle.load(
    f, encoding='iso-8859-1'
)
```

This allows the older serialized dataset to be loaded under the Python environment used for the project.


In [ ]:
import mnist_loader

training_data, validation_data, test_data = mnist_loader.load_data_wrapper()

training_data = list(training_data)
validation_data = list(validation_data)
test_data = list(test_data)

print("Training examples:", len(training_data))
print("Validation examples:", len(validation_data))
print("Test examples:", len(test_data))
print("Input shape:", training_data[0][0].shape)


Expected dataset organization from the loader:

- 50,000 training examples
- 10,000 validation examples
- 10,000 test examples
- each image represented as a `784 x 1` vector


## 3. Define the final trained network

The final training script defines:

```text
784 -> 30 -> 20 -> 10
```

This is a fully connected feed-forward network, not a convolutional neural network.

The input layer contains the 784 pixels of a 28 x 28 MNIST image. The two hidden layers contain 30 and 20 neurons. The output layer contains 10 neurons corresponding to digits 0 through 9.


In [ ]:
import network2

net = network2.Network([784, 30, 20, 10])
print(net.sizes)


## 4. Train the network

The final `trainNN.py` uses the following settings:

| Setting | Value |
|---|---:|
| Architecture | 784 -> 30 -> 20 -> 10 |
| Epochs | 100 |
| Mini-batch size | 10 |
| Learning rate (`eta`) | 0.1 |
| L2 regularization (`lambda`) | 5.0 |
| Evaluation data | MNIST validation set |
| Activation | Sigmoid |
| Cost function | Cross-entropy (`network2.py` default) |
| Saved model | `WeigntsAndBiases.txt` |

The spelling of `WeigntsAndBiases.txt` is intentional because that is the filename used by the working project.


In [ ]:
# This is the final training sequence.
# Running it retrains the model and will overwrite WeigntsAndBiases.txt.

import mnist_loader
import network2

training_data, validation_data, test_data = mnist_loader.load_data_wrapper()
training_data = list(training_data)
validation_data = list(validation_data)

net = network2.Network([784, 30, 20, 10])

net.SGD(
    training_data,
    100,                 # epochs
    10,                  # mini-batch size
    0.1,                 # learning rate
    lmbda=5.0,
    evaluation_data=validation_data,
    monitor_evaluation_accuracy=True
)

net.save("WeigntsAndBiases.txt")


### Saved-model evidence

![Saved weights and biases file](mnist_saved_model_file.png)

The saved model is JSON-formatted and stores:

- `sizes`
- `weights`
- `biases`
- the cost-function name

The final file reports the architecture `[784, 30, 20, 10]`.


In [ ]:
import json

with open("WeigntsAndBiases.txt", "r") as f:
    saved_model = json.load(f)

print("Saved architecture:", saved_model["sizes"])
print("Number of weight matrices:", len(saved_model["weights"]))
print("Number of bias vectors:", len(saved_model["biases"]))
print("Saved cost function:", saved_model["cost"])


## 5. Reload and validate the saved model

The final model can be reloaded with `network2.load()` and evaluated on the 10,000-image MNIST test set.

For the final model used in the dissertation work, the later validation run produced:

```text
9663 / 10000 = 96.63%
```

The following cell reproduces that calculation from the saved model file.


In [ ]:
import mnist_loader
import network2

_, _, test_data = mnist_loader.load_data_wrapper()
test_data = list(test_data)

net = network2.load("WeigntsAndBiases.txt")
correct = net.accuracy(test_data)

print(f"{correct}/10000 = {correct / 100:.2f}%")


## 6. Test binary/thresholded MNIST inputs

The camera implementation eventually converted selected pixels to binary-style values rather than feeding the original grayscale MNIST values directly. To examine how binarization affected the trained software model, `test_binary_mnist.py` evaluates the saved network using thresholds of:

```text
0.30, 0.40, 0.50, 0.60, 0.70
```

Each test image is converted with:

```python
binary_x = (x >= threshold).astype(np.float64)
```

The results from the final model were:

| Input | Accuracy |
|---|---:|
| Original grayscale | 96.63% |
| Threshold 0.30 | 96.09% |
| Threshold 0.40 | 96.13% |
| Threshold 0.50 | 95.74% |
| Threshold 0.60 | 94.85% |
| Threshold 0.70 | 94.02% |

This experiment was a **software-model input-format check**. It was not the FPGA hardware accuracy benchmark.


In [ ]:
import mnist_loader
import network2
import numpy as np

_, _, test_data = mnist_loader.load_data_wrapper()
test_data = list(test_data)

net = network2.load("WeigntsAndBiases.txt")

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

print("Original grayscale:")
correct = net.accuracy(test_data)
print(f"{correct}/10000 = {correct / 100:.2f}%")
print()

for threshold in thresholds:
    binary_test_data = []
    for x, y in test_data:
        binary_x = (x >= threshold).astype(np.float64)
        binary_test_data.append((binary_x, y))

    correct = net.accuracy(binary_test_data)
    print(f"Threshold {threshold:.2f}: {correct}/10000 = {correct / 100:.2f}%")


## 7. Prepare the trained parameters for ZyNet

The `genWegitsAndBias.py` reads `WeigntsAndBiases.txt` and converts the floating-point weights and biases to fixed-point two's-complement representations for ZyNet.

The conversion settings are:

```python
dataWidth = 16
dataIntWidth = 1
weightIntWidth = 4

dataFracWidth = 15
weightFracWidth = 12
biasIntWidth = 5
biasFracWidth = 11
```

The script creates:

- per-neuron weight `.mif` files under `./w_b/`
- per-neuron bias `.mif` files under `./w_b/`
- `weightValues.h`
- `biasValues.h`

### Modification to the conversion script

The working copy contains the explicit note:

```python
#Note I added the next line
wInDec = 0
```

This initialization was added before the weight-conversion loops. It prevents `wInDec` from being undefined when a very small scientific-notation weight is replaced by zero before the value is written to the generated header array.


In [ ]:
# Core fixed-point conversion settings from genWegitsAndBias.py

dataWidth = 16
dataIntWidth = 1
weightIntWidth = 4

dataFracWidth = dataWidth - dataIntWidth
weightFracWidth = dataWidth - weightIntWidth
biasIntWidth = dataIntWidth + weightIntWidth
biasFracWidth = dataWidth - biasIntWidth

print("dataFracWidth =", dataFracWidth)
print("weightFracWidth =", weightFracWidth)
print("biasIntWidth =", biasIntWidth)
print("biasFracWidth =", biasFracWidth)


To regenerate the ZyNet parameter files with the script, make sure a `w_b` directory exists and run:

```bash
python genWegitsAndBias.py
```

The notebook does not reproduce all of the generated `.mif` files inline because the script is the authoritative conversion utility.


## 8. Define and compile the pretrained ZyNet model

The original ZyNet MNIST sample in `dsdnu/zynet` defined a larger sample network:

```text
flatten(784)
Dense(30, sigmoid)
Dense(30, sigmoid)
Dense(10, sigmoid)
Dense(10, sigmoid)
Dense(10, hardmax)
```

and loaded `WeigntsAndBiasesSigmoid.txt`.

The final research model was changed to match the trained `[784, 30, 20, 10]` network:

```text
flatten(784)
Dense(30, sigmoid)
Dense(20, sigmoid)
Dense(10, sigmoid)
Dense(10, hardmax)
```

The final scripts load `WeigntsAndBiases.txt` and compile the model as pretrained.


In [ ]:
from zynet import zynet
from zynet import utils

def compile_mnist_zynet(
    dataWidth=8,
    sigmoidSize=5,
    weightIntSize=4,
    inputIntSize=1
):
    model = zynet.model()

    model.add(zynet.layer("flatten", 784))
    model.add(zynet.layer("Dense", 30, "sigmoid"))
    model.add(zynet.layer("Dense", 20, "sigmoid"))
    model.add(zynet.layer("Dense", 10, "sigmoid"))
    model.add(zynet.layer("Dense", 10, "hardmax"))

    weightArray = utils.genWeightArray("WeigntsAndBiases.txt")
    biasArray = utils.genBiasArray("WeigntsAndBiases.txt")

    model.compile(
        pretrained="Yes",
        weights=weightArray,
        biases=biasArray,
        dataWidth=dataWidth,
        weightIntSize=weightIntSize,
        inputIntSize=inputIntSize,
        sigmoidSize=sigmoidSize,
    )

    return model

# Run only in an environment where the ZyNet package/repository is available:
# model = compile_mnist_zynet()


## 9. Verified changes from the original ZyNet MNIST sample

The following differences are directly supported by the retained scripts and the upstream ZyNet sample.

| Item | Original ZyNet sample | Final research workflow |
|---|---|---|
| First hidden layer | 30 sigmoid | 30 sigmoid |
| Second hidden layer | 30 sigmoid | **20 sigmoid** |
| Additional sigmoid output-stage layer | present | **removed to match the trained 784-30-20-10 network** |
| Saved parameter filename | `WeigntsAndBiasesSigmoid.txt` | `WeigntsAndBiases.txt` |
| Example data width | invoked with `dataWidth=31` | final scripts default to **8** for ZyNet compilation |
| Weight integer width | hard-coded as 4 in sample compile call | exposed as `weightIntSize=4` |
| Input integer width | hard-coded as 1 in sample compile call | exposed as `inputIntSize=1` |
| Project-generation ending | calls ZyNet Xilinx project/IP/system generation | **not part of this model notebook** |

The `mnistZyNet.py` and `mnistZyNet2.py` later diverge further because the FPGA implementation had to coexist with the camera architecture. Those Vivado/project-integration steps are intentionally excluded here.


## 10. Why this notebook stops before Vivado

The original ZyNet sample continues from model compilation into automatic Xilinx project generation.

That approach was not used unchanged for the final research hardware because the FPGA design also had to incorporate:

- the OV7670 camera;
- image capture and preprocessing;
- stored image BRAM;
- VGA/display logic;
- later local and cloud-assisted FPGA integration.

The `mnistZyNet.py` attempts a more complete camera-aware project-generation path, while `mnistZyNet2.py` instead opens an existing camera project and injects the ZyNet module. Those scripts document the later implementation evolution, but their Vivado-specific portions belong in the FPGA implementation branch.

For the purposes of **model reproducibility**, the stopping point is therefore:

> train model -> save weights/biases -> validate model -> convert parameters -> compile the pretrained ZyNet model definition

Hardware project construction follows in the separate FPGA workflow.


## 11. Auxiliary retained conversion script

The  `convert.py` reads `WeigntsAndBiases.txt`, flattens the saved weight and bias arrays, and writes C arrays to `params.h`.

This utility is preserved with the model files because it converts the same trained model parameters, but it is **not required for the core ZyNet training workflow** documented above.

Its output is useful for software-side or Processing-System representations of the same trained model.


## 12. Final model artifacts

The key reproducibility artifacts for the FPGA/ZyNet MNIST model are:

| Artifact | Role |
|---|---|
| `mnist.pkl.gz` | retained MNIST dataset archive used by the loader |
| `mnist_loader.py` | dataset loader and reshaping |
| `network2.py` | feed-forward network, sigmoid activation, SGD/backpropagation |
| `trainNN.py` | final architecture and training settings |
| `WeigntsAndBiases.txt` | final floating-point trained model |
| `test_binary_mnist.py` | grayscale and thresholded-input validation |
| `genWegitsAndBias.py` | fixed-point/per-neuron ZyNet parameter conversion |
| `mnistZyNet.py` / `mnistZyNet2.py` | later ZyNet model compilation and FPGA-integration evolution |
| `weights.mem` | retained final FPGA weight artifact |
| `biases.mem` | retained final FPGA bias artifact |

### Final validated software-model result

The final model achieved:

```text
9663 / 10000 = 96.63%
```

on the original grayscale MNIST test set.

That software result should remain distinct from the later FPGA hardware accuracy result and from camera-based functional validation.


## Reproducibility note

Training is stochastic unless the random-number generators are explicitly seeded. Therefore, rerunning the 100-epoch training cell may produce a model with slightly different weights and validation accuracy.

For exact reproduction of the model used in the experiments, use the retained:

```text
WeigntsAndBiases.txt
```

The training cell documents how the model was created; the retained parameter file preserves the actual trained instance used by the project.
